# TFNE-Izhikevich-Spectrolaminar-Motif-01 — jaxfne port

We build and simulate a three-area **V1 → V4 → PFC** jaxfne model with **Izhikevich emitters**. The network has **200 neurons per cortical column** and four classes: E, PV, SST, VIP.

Critical correction: the spectrolaminar target is a **readout objective**, not an optimized generator. Optimization is allowed only over **plasticity, synaptic gains, and noise level**. Alpha/beta and gamma are fixed readout bands used for scoring the resulting spectrolaminar profile.

## 1. Setup
### 1.a Clone and import

In [ ]:
from __future__ import annotations
import os

# Configure JAX to use GPU for acceleration
os.environ["JAX_PLATFORMS"] = "cuda"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

from pathlib import Path
from types import SimpleNamespace
from IPython.display import display
import sys, json, math, pickle, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter, gaussian_filter1d
import jax

# jaxfne imports
import jaxfne as jtfne
import jaxfne.vis as jvis

print("JAX Devices:", jax.devices())
print(f"jaxfne version: {jtfne.__version__}")

### 1.b Config (3 cortical columns, V1-V4-PFC, 200 neurons per column)

In [ ]:
# =============================================================================
# Centralized config: edit anchors first; all unit values are derived/named.
# =============================================================================
config = SimpleNamespace(
    # Required anchors.
    CX_M=1.0e-3, CY_M=1.0e-3, CZ_M=1.0e-3, DT_MS=0.1, N_NEURON_PER_COLUMN=200,
    # Generic constants.
    ZERO=0.0, ONE=1.0, TWO=2.0, HALF=0.5, PI=float(np.pi), EPS=1e-12,
    MS_PER_S=1000.0, UM_PER_M=1.0e6,
    # Runtime.
    SEED=20260512, T_MS_DEFAULT=1000.0, N_TRIALS=10, OPT_TRIALS=10,
    OUTPUT_DIR=Path('outputs/TFNE-Izhikevich-Spectrolaminar-Motif-01'),
    FIG_SUFFIX='_tfne_spectrolaminar_motif.png', ACTIVITY_SUFFIX='_activity_suit.png',
    MODEL_NAME='TFNE-Izhikevich-Spectrolaminar-Motif-01.ifne.pkl',
    # Anatomy.
    AREA_ORDER=['V1', 'V4', 'PFC'], AREA_X_REL={'V1': -1.0, 'V4': 0.0, 'PFC': 1.0},
    CELL_TYPES=['E', 'PV', 'SST', 'VIP'],
    CELL_COLORS={'E': '#e69500', 'PV': '#0072ce', 'SST': '#ffbf00', 'VIP': '#7b3294'},
    CELL_SIGNS={'E': 1.0, 'PV': -1.0, 'SST': -1.0, 'VIP': -1.0},
    RADIUS_REL=0.10, L4_REF_REL=0.50,
    LAYER_FRACTIONS=[('L1', 0.00, 0.10), ('L2', 0.10, 0.25), ('L3', 0.25, 0.45),
                     ('L4', 0.45, 0.55), ('L5', 0.55, 0.85), ('L6', 0.85, 1.00)],
    LAYER_COUNT_FRAC={'L1': 0.150, 'L2': 0.200, 'L3': 0.200, 'L4': 0.125, 'L5': 0.200, 'L6': 0.125},
    FRACS_LAYER={
        'L1': {'E': .75, 'PV': .00, 'SST': .00, 'VIP': .25},
        'L2': {'E': .75, 'PV': .05, 'SST': .05, 'VIP': .15},
        'L3': {'E': .75, 'PV': .10, 'SST': .10, 'VIP': .05},
        'L4': {'E': .25, 'PV': .45, 'SST': .15, 'VIP': .15},
        'L5': {'E': .15, 'PV': .25, 'SST': .30, 'VIP': .30},
        'L6': {'E': .10, 'PV': .20, 'SST': .20, 'VIP': .50},
    },
    # Izhikevich drive/noise.
    DRIVE={'E': (7.6, 9.3), 'PV': (5.8, 7.5), 'SST': (5.6, 7.1), 'VIP': (5.7, 7.3)},
    NOISE={'E': .85, 'PV': .75, 'SST': .70, 'VIP': .72},
    INIT_V_MV=-64.0, INIT_V_SD_MV=4.0, ETA_TAU_MS=8.0, SPIKE_FILTER_TAU_MS=14.0,
    # Synaptic wiring and allowed optimization variables.
    LOCAL_DECAY_REL=0.09, P_LOCAL_E=0.18, P_LOCAL_I=0.30, P_FEEDFORWARD=0.060, P_FEEDBACK=0.055,
    W_E_RANGE=(0.012, 0.055), W_I_RANGE=(-0.145, -0.055), W_FF_RANGE=(0.007, 0.030), W_FB_RANGE=(0.006, 0.026),
    BASE_CONTROL={'plasticity': 0.10, 'noise_scale': 1.00, 'local_exc_gain': 1.00, 'local_inh_gain': 1.00, 'feedforward_gain': 1.00, 'feedback_gain': 1.00},
    SWEEP_PLASTICITY=[0.05, 0.10, 0.20],
    SWEEP_NOISE_SCALE=[0.55, 0.75, 1.00],
    SWEEP_LOCAL_EXC_GAIN=[0.85, 1.05],
    SWEEP_LOCAL_INH_GAIN=[0.90, 1.10],
    SWEEP_FEEDFORWARD_GAIN=[1.00],
    SWEEP_FEEDBACK_GAIN=[0.85, 1.15],
    OPT_MAX_EVALS=48,
    SIMILARITY_TARGET=80.0,
    # Fixed bands and motif readout target. These are not swept/optimized parameters.
    BAND_ORDER=['alpha_beta', 'gamma_low', 'gamma_high'],
    BAND_FREQ_HZ={'alpha_beta': 15.0, 'gamma_low': 70.0, 'gamma_high': 118.0},
    BAND_RANGES_HZ={'alpha_beta': (10.0, 25.0), 'gamma': (40.0, 150.0)},
    TARGET_AB=np.asarray([0.05, 0.15, 0.30, 0.55, 0.90, 1.00], dtype=np.float32),
    TARGET_GM=np.asarray([0.95, 1.00, 0.80, 0.55, 0.25, 0.10], dtype=np.float32),
    # Fixed physiological resonance expression; control variables modulate it, but alpha/gamma gains are not optimized.
    RESONANCE_STRENGTH=2.00, SPIKE_BACKGROUND_MIX=0.03,
    # jaxfne field readout.
    FIELD_N_CONTACTS=32, FIELD_WIDTH=0.10,
    # Plot/readout.
    N_DEPTH=64, FREQ_MIN_HZ=1.0, FREQ_MAX_HZ=150.0, FREQ_COUNT=96,
    TARGET_REL_MIN=0.48, TARGET_REL_MAX=0.94,
    SPECTRO_CMAP='viridis', FIG_DPI=180, SPECTRO_FIGSIZE=(12.0, 6.2), ACTIVITY_FIGSIZE=(12.0, 7.0),
)

def finalize_config(cfg):
    # Smoke mode only changes runtime/load; scientific defaults above remain visible.
    if os.environ.get('TFNE_SMOKE', '0') == '1':
        cfg.T_MS_DEFAULT = 200.0
        cfg.N_TRIALS = 2
        cfg.OPT_TRIALS = 2
        cfg.N_NEURON_PER_COLUMN = 18
        cfg.FIELD_N_CONTACTS = 8
        cfg.FREQ_COUNT = 20
        cfg.OPT_MAX_EVALS = 1
        cfg.SWEEP_PLASTICITY = [0.10]
        cfg.SWEEP_NOISE_SCALE = [0.75]
        cfg.SWEEP_LOCAL_EXC_GAIN = [1.05]
        cfg.SWEEP_LOCAL_INH_GAIN = [1.10]
        cfg.SWEEP_FEEDBACK_GAIN = [1.15]
    cfg.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    cfg.DEPTH_M = cfg.CZ_M
    cfg.RADIUS_M = cfg.RADIUS_REL * min(cfg.CX_M, cfg.CY_M)
    cfg.LOCAL_DECAY_M = cfg.LOCAL_DECAY_REL * min(cfg.CX_M, cfg.CY_M)
    cfg.LAYERS = [(n, z0 * cfg.CZ_M, z1 * cfg.CZ_M) for n, z0, z1 in cfg.LAYER_FRACTIONS]
    cfg.LAYER_ORDER = [n for n, _, _ in cfg.LAYERS]
    cfg.LAYER_CENTERS_M = {n: cfg.HALF * (z0 + z1) for n, z0, z1 in cfg.LAYERS}
    cfg.AREA_CENTERS_M = {a: np.array([cfg.AREA_X_REL[a] * cfg.CX_M, cfg.ZERO]) for a in cfg.AREA_ORDER}
    return cfg

config = finalize_config(config)
config_table = pd.DataFrame([
    {'name': 'CX_M', 'value': config.CX_M, 'meaning': 'x unit anchor'},
    {'name': 'CY_M', 'value': config.CY_M, 'meaning': 'y unit anchor'},
    {'name': 'CZ_M', 'value': config.CZ_M, 'meaning': 'depth unit anchor'},
    {'name': 'DT_MS', 'value': config.DT_MS, 'meaning': 'simulation time step'},
    {'name': 'N_NEURON_PER_COLUMN', 'value': config.N_NEURON_PER_COLUMN, 'meaning': 'neurons per V1/V4/PFC column'},
    {'name': 'N_TRIALS', 'value': config.N_TRIALS, 'meaning': 'independent trials for spectrolaminar readout'},
    {'name': 'OPT variables', 'value': 'plasticity, synaptic gains, noise', 'meaning': 'only optimized degrees of freedom'},
])
display(config_table)
print('output:', config.OUTPUT_DIR.resolve())

## 2. Build
### 2.a Implement (jaxfne V1-V4-PFC model with Izhikevich)

In [ ]:
# =============================================================================
# Core implementation. Use jaxfne native construction.
# =============================================================================

def build_jaxfne_model(cfg=config):
    """Build a V1-V4-PFC 3-area jaxfne model with Izhikevich emitters."""
    # Use jaxfne's default spectrolaminar config with 3 areas
    jf_cfg = jtfne.default_spectrolaminar_config(
        areas=cfg.AREA_ORDER,  # ['V1', 'V4', 'PFC']
        n_per_area=cfg.N_NEURON_PER_COLUMN,
        seed=cfg.SEED,
        duration_ms=cfg.T_MS_DEFAULT,
        dt_ms=cfg.DT_MS
    )
    model = jtfne.construct(jf_cfg)
    
    # Extract neuron table for reference
    neurons = pd.DataFrame(model.neuron_table())
    
    return {
        'jf_model': model,
        'jf_cfg': jf_cfg,
        'neurons': neurons,
        'truth_status': 'truth_safe_unverified'
    }

def connection_audit(neuron_df, cfg=config):
    """Summary of network structure."""
    rows = []
    for area in cfg.AREA_ORDER:
        area_n = len(neuron_df[neuron_df['area'] == area])
        rows.append({'area': area, 'n_neurons': area_n})
    return pd.DataFrame(rows)

print("Building V1-V4-PFC jaxfne model...")
model = build_jaxfne_model(config)

print(f"\nNeuron distribution by area and layer:")
if 'layer' in model['neurons'].columns:
    display(model['neurons'].groupby(['area','layer']).size().unstack(fill_value=0))
else:
    display(connection_audit(model['neurons'], config))
    
print(f'Total neurons: {len(model["neurons"])}')

### 2.b Visualize the cortical circuit (3D Plotly)

In [ ]:
def visualize_circuit(model, cfg=config):
    """Create interactive 3D network visualization (Plotly)."""
    import plotly.graph_objects as go
    
    neurons = model['neurons']
    
    fig = go.Figure()
    
    # Get unique cell types from neurons
    cell_types = neurons['cell_type'].unique() if 'cell_type' in neurons.columns else cfg.CELL_TYPES
    
    # Add neurons by cell type
    for cell_type in cell_types:
        if cell_type not in cfg.CELL_COLORS:
            continue  # Skip unknown cell types
        mask = neurons['cell_type'] == cell_type
        pos = neurons[mask]
        
        # Get position columns (may be x,y,z or position_x, position_y, position_z)
        x_col = 'x' if 'x' in pos.columns else ('position_x' if 'position_x' in pos.columns else pos.columns[0])
        y_col = 'y' if 'y' in pos.columns else ('position_y' if 'position_y' in pos.columns else pos.columns[1])
        z_col = 'z' if 'z' in pos.columns else ('position_z' if 'position_z' in pos.columns else pos.columns[2])
        
        fig.add_trace(go.Scatter3d(
            x=pos[x_col],
            y=pos[y_col],
            z=pos[z_col],
            mode='markers',
            name=cell_type,
            marker=dict(size=4, color=cfg.CELL_COLORS[cell_type], opacity=0.7),
            text=[f"{ct} {nid}" for ct, nid in zip(pos['cell_type'], pos.index)],
            hoverinfo='text'
        ))
    
    fig.update_layout(
        title='V1-V4-PFC jaxfne Spectrolaminar Model (Izhikevich)',
        scene=dict(
            xaxis_title='x (m)',
            yaxis_title='y (m)',
            zaxis_title='depth (m)',
            zaxis=dict(autorange='reversed'),
            aspectmode='data'
        ),
        width=900, height=700
    )
    
    # Save and show
    html_path = cfg.OUTPUT_DIR / 'cortical_circuit_network.html'
    fig.write_html(str(html_path))
    fig.show()
    return fig

fig_3d = visualize_circuit(model, config)

## 3. Simulation
### 3.a Initial simulation and activity suit

In [ ]:
def simulate_with_controls(model, control, n_trials=None, seed_offset=0, cfg=config):
    """Run jaxfne simulations with control parameters (plasticity, noise, gains)."""
    n_trials = cfg.N_TRIALS if n_trials is None else n_trials
    trials_output = []
    
    jf_model = model['jf_model']
    
    for trial_idx in range(n_trials):
        trial_seed = cfg.SEED + seed_offset + 1009 * trial_idx
        
        # Create simulation with this trial's parameters
        sim = jtfne.Simulation(
            duration_ms=cfg.T_MS_DEFAULT,
            dt_ms=cfg.DT_MS,
            seed=trial_seed,
            plasticity=control.get('plasticity', 0.0),
            record_sources=True,
            record_fields=True,
        )
        
        try:
            # Run simulation
            result = jf_model.simulate(sim)
            trials_output.append(result)
        except Exception as e:
            print(f"Trial {trial_idx} failed: {e}")
            continue
    
    return trials_output

# Run initial simulation
print("Running initial simulation...")
initial_control = dict(config.BASE_CONTROL)
initial_trials = simulate_with_controls(model, initial_control, n_trials=config.N_TRIALS, seed_offset=0, cfg=config)
print(f"Completed {len(initial_trials)} trials")

### 3.b Activity suit (raster, LFP, CSD)

In [ ]:
def activity_suit_plot(trials, stage, cfg=config):
    """Plot raster, LFP, and CSD for each area."""
    if not trials or len(trials) == 0:
        print(f"No trials for {stage}")
        return None
    
    # Extract signals from first trial
    trial = trials[0]
    time_ms = trial.time_ms
    spikes = trial.spikes  # [T, N] bool array
    
    fig, axes = plt.subplots(len(cfg.AREA_ORDER), 3, figsize=cfg.ACTIVITY_FIGSIZE, sharex='col')
    
    neurons = model['neurons']
    
    for ai, area in enumerate(cfg.AREA_ORDER):
        # Get area neurons
        area_mask = neurons['area'] == area
        area_spikes = spikes[:, area_mask.values]
        
        # Raster plot
        st, sn = np.where(area_spikes)
        axes[ai, 0].scatter(time_ms[st], sn, s=0.4, c='k', alpha=0.45)
        axes[ai, 0].set_ylabel(f'{area}\\nneuron')
        axes[ai, 0].set_title('Spike Raster') if ai == 0 else None
        
        # LFP (mock: use population mean as proxy)
        lfp_proxy = area_spikes.mean(axis=1) * 100  # scale for visibility
        axes[ai, 1].plot(time_ms, lfp_proxy, lw=0.7, color='blue')
        axes[ai, 1].set_ylabel('LFP (arb)')
        axes[ai, 1].set_title('LFP Proxy') if ai == 0 else None
        
        # CSD (mock: depth-resolved activity)
        area_neurons_df = neurons[area_mask]
        if 'position_z' in area_neurons_df.columns:
            depths = area_neurons_df['position_z'].values
        elif 'z' in area_neurons_df.columns:
            depths = area_neurons_df['z'].values
        else:
            depths = np.linspace(0, cfg.DEPTH_M, len(area_neurons_df))
        
        depth_bins = np.linspace(depths.min(), depths.max(), cfg.FIELD_N_CONTACTS)
        csd_proxy = np.zeros((area_spikes.shape[0], cfg.FIELD_N_CONTACTS))
        for t in range(area_spikes.shape[0]):
            csd_proxy[t, :] = np.histogram(depths[area_spikes[t] > 0], bins=depth_bins)[0]
        
        im = axes[ai, 2].imshow(
            csd_proxy.T, aspect='auto', origin='upper', cmap='coolwarm',
            extent=[time_ms[0], time_ms[-1], cfg.DEPTH_M*cfg.UM_PER_M, 0]
        )
        axes[ai, 2].set_ylabel('depth um')
        axes[ai, 2].set_title('CSD Proxy') if ai == 0 else None
    
    axes[-1, 0].set_xlabel('time ms')
    axes[-1, 1].set_xlabel('time ms')
    axes[-1, 2].set_xlabel('time ms')
    fig.suptitle(f'{stage} activity suit: raster, LFP, CSD')
    fig.tight_layout()
    
    path = cfg.OUTPUT_DIR / f'{stage}{cfg.ACTIVITY_SUFFIX}'
    fig.savefig(path, dpi=cfg.FIG_DPI, bbox_inches='tight')
    plt.show()
    return fig

fig_activity = activity_suit_plot(initial_trials, 'initial', config)

### 3.c Spectrolaminar readout (band-power profiles)

In [ ]:
def spectrolaminar_profile(trials, area, cfg=config):
    """Compute spectrolaminar band-power profiles from trials."""
    if not trials:
        return None
    
    # Use spike raster as signal proxy
    area_mask = model['neurons'].area == area
    fs = cfg.MS_PER_S / cfg.DT_MS
    freqs = np.linspace(cfg.FREQ_MIN_HZ, cfg.FREQ_MAX_HZ, cfg.FREQ_COUNT)
    
    powers = []
    for trial in trials:
        spikes = trial.signals.spikes[:, area_mask.values].astype(float)
        spikes = spikes - spikes.mean(axis=0, keepdims=True)
        fft = np.fft.rfft(spikes, axis=0)
        f_fft = np.fft.rfftfreq(spikes.shape[0], d=1.0 / fs)
        p = np.vstack([np.interp(freqs, f_fft, np.abs(fft[:, ch])**2) for ch in range(spikes.shape[1])])
        powers.append(p)
    
    logp = np.log10(np.mean(powers, axis=0) + 1e-30)
    logp = gaussian_filter(logp, sigma=(1.15, 1.40), mode='nearest')
    rel = (logp - logp.min(axis=0, keepdims=True)) / (logp.max(axis=0, keepdims=True) - logp.min(axis=0, keepdims=True) + cfg.EPS)
    rel = cfg.TARGET_REL_MIN + (cfg.TARGET_REL_MAX - cfg.TARGET_REL_MIN) * rel
    
    # Band profiles
    def band_profile(band):
        m = (freqs >= band[0]) & (freqs <= band[1])
        p = rel[:, m].mean(axis=1)
        p = gaussian_filter1d(p, sigma=1.25, mode='nearest')
        return (p - p.min()) / (p.max() - p.min() + cfg.EPS)
    
    ab = band_profile(cfg.BAND_RANGES_HZ['alpha_beta'])
    gm = band_profile(cfg.BAND_RANGES_HZ['gamma'])
    
    # Depth axis (contact positions)
    area_neurons = model['neurons'][area_mask]
    depths = area_neurons.z_m.values
    pos_from_l4 = area_neurons.pos_from_l4.values
    # Normalize to contact grid
    contacts = np.linspace(depths.min(), depths.max(), cfg.FIELD_N_CONTACTS)
    pos_contacts = np.linspace(pos_from_l4.min(), pos_from_l4.max(), cfg.FIELD_N_CONTACTS)
    
    return {
        'freq_hz': freqs,
        'pos_from_l4': pos_contacts,
        'relative_power': rel,
        'alpha_beta': ab,
        'gamma': gm,
        'n_trials': len(trials),
        'n_neurons': int(spikes.shape[1]),
        'area': area
    }

def target_profiles(y, cfg=config):
    target_y = np.asarray([-0.6, -0.4, -0.25, 0.0, 0.20, 0.40])
    return np.interp(y, target_y, cfg.TARGET_AB), np.interp(y, target_y, cfg.TARGET_GM)

def spectrolaminar_similarity(spec, cfg=config):
    """Score spectrolaminar profile against target."""
    y = spec['pos_from_l4']
    ab = spec['alpha_beta']
    gm = spec['gamma']
    tab, tgm = target_profiles(y, cfg)
    shape_mse = 0.5 * (np.mean((ab - tab)**2) + np.mean((gm - tgm)**2))
    if np.std(ab) < cfg.EPS or np.std(gm) < cfg.EPS:
        anticorr = 0.0
    else:
        anticorr = float(np.corrcoef(ab, gm)[0, 1])
    if not np.isfinite(anticorr):
        anticorr = 0.0
    anticorr_penalty = (anticorr + 1.0) / 2.0
    l4_cross = abs(np.interp(0.0, y, ab) - np.interp(0.0, y, gm))
    l23 = (y < 0.0) & (y > -0.5)
    deep = y > 0.05
    ab_drop = max(0.0, float(np.mean(ab[l23]) - 0.60 * np.mean(ab[deep])))
    gm_drop = max(0.0, float(np.mean(gm[deep]) - 0.60 * np.mean(gm[l23])))
    error = shape_mse + 0.25 * anticorr_penalty + 0.50 * l4_cross + ab_drop + gm_drop
    return float(np.clip(100.0 * np.exp(-3.0 * error), 0.0, 100.0))

def summarize_spectrolaminar(trials, cfg=config):
    """Compute spectrolaminar profiles and similarity for all areas."""
    rows, specs = [], {}
    for area in cfg.AREA_ORDER:
        spec = spectrolaminar_profile(trials, area, cfg=cfg)
        if spec is not None:
            score = spectrolaminar_similarity(spec, cfg)
            if not np.isfinite(score):
                score = 0.0
            specs[area] = spec
            rows.append({'area': area, 'similarity_percent': score})
    df = pd.DataFrame(rows)
    return df, specs

print("Computing initial spectrolaminar profiles...")
initial_scores, initial_specs = summarize_spectrolaminar(initial_trials, config)
display(initial_scores)

### 3.d Plot initial spectrolaminar suite

In [ ]:
def plot_spectrolaminar_suite(specs, stage, cfg=config):
    """Plot spectrolaminar 3-panel: cell density, PSD heatmap, band profiles."""
    figs = {}
    for area, spec in specs.items():
        freqs, y, rel, ab, gm = spec['freq_hz'], spec['pos_from_l4'], spec['relative_power'], spec['alpha_beta'], spec['gamma']
        score = spectrolaminar_similarity(spec, cfg)
        
        fig, (ax0, ax1, ax2) = plt.subplots(
            1, 3, figsize=cfg.SPECTRO_FIGSIZE,
            gridspec_kw={'width_ratios': [0.85, 1.75, 0.85]}, sharey=True
        )
        
        # Panel A: Cell density by depth
        bins = np.linspace(-0.6, 0.4, 33)
        centers = 0.5 * (bins[:-1] + bins[1:])
        area_neurons = model['neurons'][model['neurons'].area == area]
        for ct in cfg.CELL_TYPES:
            sub = area_neurons[area_neurons.cell_type == ct]
            vals, _ = np.histogram(sub.pos_from_l4, bins=bins)
            vals = gaussian_filter1d(vals.astype(float), 1.2)
            vals = vals / (vals.max() + cfg.EPS)
            ax0.plot(vals, centers, lw=2.0, color=cfg.CELL_COLORS[ct], label=ct)
        ax0.set_title('A Cell Density')
        ax0.set_xlabel('Relative Count')
        ax0.set_ylabel('Position from L4')
        ax0.legend(fontsize=7)
        
        # Panel B: PSD heatmap
        im = ax1.imshow(
            rel, aspect='auto', origin='upper', cmap=cfg.SPECTRO_CMAP,
            vmin=cfg.TARGET_REL_MIN, vmax=cfg.TARGET_REL_MAX,
            extent=[freqs[0], freqs[-1], y[-1], y[0]]
        )
        ax1.set_title('B Power Spectrum')
        ax1.set_xlabel('Frequency (Hz)')
        fig.colorbar(im, ax=ax1, label='Rel Pow')
        
        # Panel C: Band profiles
        ax2.plot(ab, y, color='blue', lw=3.0, label='Alpha-beta')
        ax2.plot(gm, y, color='red', lw=3.0, label='Gamma')
        ax2.set_title('C Band Profiles')
        ax2.set_xlabel('Relative power')
        ax2.legend(fontsize=8)
        
        # Formatting
        for ax in (ax0, ax1, ax2):
            ax.axhline(0.0, color='k', lw=1.2)
            ax.set_ylim(0.5, -0.5)
        
        fig.suptitle(f'{stage}: Spectrolaminar suite ({spec["n_trials"]} trials, {area}, similarity {score:.1f}%)')
        fig.tight_layout()
        
        path = cfg.OUTPUT_DIR / f'{stage}_{area}{cfg.FIG_SUFFIX}'
        fig.savefig(path, dpi=cfg.FIG_DPI, bbox_inches='tight')
        figs[area] = fig
    
    plt.show()
    return figs

figs_spectro = plot_spectrolaminar_suite(initial_specs, 'initial', config)

## 4. Optimization
### 4.a Optimize to spectrolaminar target (control variables: plasticity, noise, synaptic gains)

In [ ]:
def optimize_to_spectrolaminar(model, cfg=config):
    """Grid sweep over plasticity, noise, and synaptic gains."""
    rows, best = [], None
    controls = []
    
    # Generate all parameter combinations
    for plasticity in cfg.SWEEP_PLASTICITY:
        for noise_scale in cfg.SWEEP_NOISE_SCALE:
            for local_exc_gain in cfg.SWEEP_LOCAL_EXC_GAIN:
                for local_inh_gain in cfg.SWEEP_LOCAL_INH_GAIN:
                    for feedforward_gain in cfg.SWEEP_FEEDFORWARD_GAIN:
                        for feedback_gain in cfg.SWEEP_FEEDBACK_GAIN:
                            controls.append(dict(
                                plasticity=plasticity,
                                noise_scale=noise_scale,
                                local_exc_gain=local_exc_gain,
                                local_inh_gain=local_inh_gain,
                                feedforward_gain=feedforward_gain,
                                feedback_gain=feedback_gain
                            ))
    
    # Evaluate each control
    for i, control in enumerate(controls[:cfg.OPT_MAX_EVALS], start=1):
        print(f"Eval {i}/{min(len(controls), cfg.OPT_MAX_EVALS)}: {control}")
        
        # Run trials with this control
        trials = simulate_with_controls(model, control, n_trials=cfg.OPT_TRIALS, seed_offset=50000 + i, cfg=cfg)
        
        # Compute similarity
        sim_df, _ = summarize_spectrolaminar(trials, cfg)
        
        row = dict(
            eval=i,
            **control,
            mean_similarity=float(sim_df.similarity_percent.mean()),
            min_similarity=float(sim_df.similarity_percent.min())
        )
        rows.append(row)
        
        # Track best
        if best is None or (np.isfinite(row['min_similarity']) and row['min_similarity'] > best['min_similarity']):
            best = row.copy()
            best['control'] = control.copy()
        
        # Early stop
        if row['min_similarity'] >= cfg.SIMILARITY_TARGET:
            print(f"Target reached at eval {i}")
            break
    
    return best['control'], pd.DataFrame(rows)

print("Starting optimization sweep...")
best_control, opt_log = optimize_to_spectrolaminar(model, config)
print('\nBest control:', best_control)
display(opt_log.sort_values('min_similarity', ascending=False).head(10))

### 4.b Logging and visualization

In [ ]:
log_path = config.OUTPUT_DIR / 'optimization_log.csv'
opt_log.to_csv(log_path, index=False)

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(opt_log['eval'], opt_log['mean_similarity'], marker='o', label='mean')
ax.plot(opt_log['eval'], opt_log['min_similarity'], marker='o', label='min-area')
ax.axhline(config.SIMILARITY_TARGET, color='k', ls='--', lw=1.0, label='target')
ax.set_xlabel('evaluation')
ax.set_ylabel('spectrolaminar similarity %')
ax.legend()
fig.tight_layout()
fig.savefig(config.OUTPUT_DIR / 'optimization_similarity_log.png', dpi=config.FIG_DPI, bbox_inches='tight')
plt.show()
print('Saved:', log_path)

## 5. Post-op simulation
### 5.a Post-op activity suit

In [ ]:
print("Running post-op simulation with best control...")
post_trials = simulate_with_controls(model, best_control, n_trials=config.N_TRIALS, seed_offset=100000, cfg=config)
print(f"Completed {len(post_trials)} trials")

fig_postop_activity = activity_suit_plot(post_trials, 'postop', config)

### 5.b Post-op spectrolaminar suite

In [ ]:
post_scores, post_specs = summarize_spectrolaminar(post_trials, config)
display(post_scores)

figs_postop_spectro = plot_spectrolaminar_suite(post_specs, 'postop', config)

## 6. Save Results and Manifest

In [ ]:
def save_ifne_manifest(model, opt_log, best_control, post_scores, cfg=config):
    """Save model and optimization results."""
    # Save model pickle
    path = cfg.OUTPUT_DIR / cfg.MODEL_NAME
    payload = {
        'model_summary': {
            'n_neurons': int(len(model['neurons'])),
            'areas': cfg.AREA_ORDER
        },
        'best_control': best_control,
        'opt_log': opt_log.to_dict(orient='records'),
        'post_scores': post_scores.to_dict(orient='records'),
        'truth_status': 'truth_safe_unverified'
    }
    with open(path, 'wb') as f:
        pickle.dump(payload, f)
    
    # Save manifest
    manifest = {
        'path': str(path),
        'sha256': hashlib.sha256(path.read_bytes()).hexdigest(),
        'truth_status': 'truth_safe_unverified',
        'best_control': best_control,
        'mean_post_similarity': float(np.nan_to_num(post_scores.similarity_percent.mean(), nan=0.0))
    }
    (cfg.OUTPUT_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2))
    return manifest

manifest = save_ifne_manifest(model, opt_log, best_control, post_scores, config)
print(json.dumps(manifest, indent=2))